In [1]:
# Optional Colab mount: uncomment if running on Colab.
# from google.colab import drive
# drive.mount("/content/drive")

import os
import subprocess

WORK_DIR = os.environ.get("WORK_DIR", os.getcwd())
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

WORK_DIR = os.path.abspath(".")
MASKSDM_DIR = os.path.abspath("../MaskSDM-MEE")
DATA_DIR = WORK_DIR
CKPT_DIR = WORK_DIR
RESULTS_DIR = os.path.join(WORK_DIR, "results_masksdm_butterfly_1000_epochs_seed1339")

os.makedirs(RESULTS_DIR, exist_ok=True)

if not os.path.exists(MASKSDM_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/zbirobin/MaskSDM-MEE", MASKSDM_DIR],
        check=True,
    )
    print("Repo cloned to", MASKSDM_DIR)
else:
    print("Repo already present, skipping.")

!pip install verde schedulefree elapid torcheval -q

os.chdir(MASKSDM_DIR)

print("\nSetup complete.")
print(f"  cwd:          {os.getcwd()}")
print(f"  MaskSDM repo: {MASKSDM_DIR}")
print(f"  work/data:    {DATA_DIR}")
print(f"  ckpts:        {CKPT_DIR}")
print(f"  results:      {RESULTS_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo already present, skipping.

Setup complete.
  cwd:          /content/drive/MyDrive/CISO/MaskSDM-MEE
  MaskSDM repo: /content/drive/MyDrive/CISO/MaskSDM-MEE
  work/data:    /content/drive/MyDrive/CISO/eButterfly
  ckpts:        /content/drive/MyDrive/CISO/eButterfly
  results:      /content/drive/MyDrive/CISO/eButterfly/results_masksdm_butterfly_1000_epochs_seed1339


In [2]:
import sys
sys.path.insert(0, MASKSDM_DIR)

import numpy as np
import pandas as pd
import json
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import spearmanr

from data_helpers import get_torch_dataset
from modules import get_model
from training_helpers import seed_everything, train

!pip install --upgrade --force-reinstall "scikit-learn>=1.6" -q
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Imports OK')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 79.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
elapid 1.0.3 requires scikit-learn<1.6,>=1.2, but you have scikit-learn 1.8.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
Device: cuda
Imports OK


## 1. Load Your Data

In [3]:
# ── Identical to CISO butterfly notebook ──────────────────────────────────────
DATA_CSV    = f'{DATA_DIR}/ebutterfly_na_2011_2025.csv'
SPLITS_JSON = f'{DATA_DIR}/ebutterfly_splits.json'

for f in [DATA_CSV, SPLITS_JSON]:
    print('OK  ' if os.path.exists(f) else 'MISSING  ', f)

butterflies = pd.read_csv(DATA_CSV)
butterflies = butterflies.reset_index(drop=True)

with open(SPLITS_JSON) as f:
    butterfly_splits = json.load(f)

print(f'\nLoaded {len(butterflies):,} rows x {len(butterflies.columns):,} columns')
print(f'Split keys: {list(butterfly_splits.keys())}')
butterflies.head(2)


OK   /content/drive/MyDrive/CISO/eButterfly/ebutterfly_na_2011_2025.csv
OK   /content/drive/MyDrive/CISO/eButterfly/ebutterfly_splits.json

Loaded 17,077 rows x 191 columns
Split keys: ['num_rows', 'meta', 'train', 'val', 'test']


,time,latitude,longitude,env_t2m_min,env_t2m_max,env_t2m_mean,env_tp_sum,env_ndvi,env_evi,env_soil_bdod,...,Tharsalea hyllus,Thymelicus lineola,Urbanus dorantes,Urbanus procne,Urbanus proteus,Vanessa atalanta,Vanessa cardui,Vanessa virginiensis,Vernia verna,Zerene cesonia
0,2025-05-16,44.481715,-77.682266,286.97748,299.25006,291.72070,5.594984e-03,0.6052,0.3350,118.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2015-04-19,34.046186,-118.275999,286.35144,299.09650,292.28778,5.420297e-07,0.1364,0.0748,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
# ── Identical column groups to CISO butterfly notebook ───────────────────────
env_cols     = [c for c in butterflies.columns if c.startswith('env_')]
species_cols = [c for c in butterflies.columns
                if c not in env_cols + ['time', 'latitude', 'longitude']]

# eButterfly daily set has no bioclim. Group env into:
#   - "worldclim_cols" (variable name kept for downstream compatibility):
#       ERA5 daily t2m/tp + MODIS NDVI/EVI  (dynamic, 6 cols)
#   - "soilgrid_cols": SoilGrids + DEM      (static, 9 cols)
soilgrid_cols  = [c for c in env_cols if c.startswith('env_soil') or c == 'env_dem']
worldclim_cols = [c for c in env_cols if c not in soilgrid_cols]

print(f'Climate/dynamic cols ({len(worldclim_cols)}): {worldclim_cols}')
print(f'Static cols          ({len(soilgrid_cols)}):  {soilgrid_cols}')
print(f'Species cols         ({len(species_cols)}):   first 5 = {species_cols[:5]}')


Climate/dynamic cols (6): ['env_t2m_min', 'env_t2m_max', 'env_t2m_mean', 'env_tp_sum', 'env_ndvi', 'env_evi']
Static cols          (9):  ['env_soil_bdod', 'env_soil_cec', 'env_soil_cfvo', 'env_soil_clay', 'env_soil_nitrogen', 'env_soil_phh2o', 'env_soil_sand', 'env_soil_silt', 'env_dem']
Species cols         (173):   first 5 = ['Abaeis nicippe', 'Aglais milberti', 'Amblyscirtes hegon', 'Amblyscirtes vialis', 'Anartia fatima']


## 2. Build Split Indices and Data Arrays

In [5]:
# ── Identical split logic to CISO butterfly notebook ─────────────────────────
train_split = np.array(butterfly_splits['train'])
val_split   = np.array(butterfly_splits['val'])
test_split  = np.array(butterfly_splits['test'])

assert train_split.max() < len(butterflies), 'Train index out of bounds'
assert val_split.max()   < len(butterflies), 'Val index out of bounds'
assert test_split.max()  < len(butterflies), 'Test index out of bounds'
print(f'train={len(train_split):,}  val={len(val_split):,}  test={len(test_split):,}')
print('All indices in bounds.')


train=13,825  val=950  test=2,302
All indices in bounds.


In [6]:
# ── Filter species: >=100 occurrences (identical threshold to CISO notebook) ─
targets_full   = butterflies[species_cols].to_numpy().astype(np.float32)
species_counts = targets_full.sum(axis=0)
keep           = species_counts >= 100
targets        = targets_full[:, keep]
species_cols_filtered = [s for s, k in zip(species_cols, keep) if k]

print(f'Species before filtering: {len(species_cols):,}')
print(f'Species after  filtering: {len(species_cols_filtered):,}')

# ── Tabular env features ──────────────────────────────────────────────────────
tabular_x = butterflies[env_cols].to_numpy().astype(np.float32)

# ── SatCLIP embeddings: not available, use zeros (masked out during training) ─
# MaskSDM's extra_masking will mask these just like real predictors.
N_SATCLIP = 256
satclip_embeddings = np.zeros((len(butterflies), N_SATCLIP), dtype=np.float32)

# ── Build data dict in MaskSDM format ────────────────────────────────────────
data = {
    'tabular_x':          tabular_x,
    'y':                  targets,
    'satclip_embeddings': satclip_embeddings,
}

# ── Per-split arrays ──────────────────────────────────────────────────────────
data['x_train'] = tabular_x[train_split]
data['x_val']   = tabular_x[val_split]
data['x_test']  = tabular_x[test_split]
data['y_train'] = targets[train_split]
data['y_val']   = targets[val_split]
data['y_test']  = targets[test_split]
data['satclip_embeddings_train'] = satclip_embeddings[train_split]
data['satclip_embeddings_val']   = satclip_embeddings[val_split]
data['satclip_embeddings_test']  = satclip_embeddings[test_split]

# ── Normalise using train statistics only ─────────────────────────────────────
train_mean = np.nanmean(data['x_train'], axis=0)
train_std  = np.nanstd(data['x_train'],  axis=0)
data['x_train'] = (data['x_train'] - train_mean) / (train_std + 1e-4)
data['x_val']   = (data['x_val']   - train_mean) / (train_std + 1e-4)
data['x_test']  = (data['x_test']  - train_mean) / (train_std + 1e-4)

print(f'x_train: {data["x_train"].shape}')
print(f'x_val:   {data["x_val"].shape}')
print(f'x_test:  {data["x_test"].shape}')


Species before filtering: 173
Species after  filtering: 171
x_train: (13825, 15)
x_val:   (950, 15)
x_test:  (2302, 15)


## 3. Verify Integrity

In [7]:
n_features = data['x_train'].shape[1]
n_species  = data['y_train'].shape[1]

# Species that appear in all three splits — used for evaluation
indices_evaluated = np.intersect1d(
    np.intersect1d(
        data['y_train'].sum(axis=0).nonzero()[0],
        data['y_val'].sum(axis=0).nonzero()[0]
    ),
    data['y_test'].sum(axis=0).nonzero()[0]
).tolist()

print(f'n_features:        {n_features}')
print(f'n_species:         {n_species}')
print(f'species_evaluated: {len(indices_evaluated)}')
print(f'train rows:        {len(data["x_train"]):,}')
print(f'val rows:          {len(data["x_val"]):,}')
print(f'test rows:         {len(data["x_test"]):,}')
print('\nAll checks passed.')

n_features:        15
n_species:         171
species_evaluated: 144
train rows:        13,825
val rows:          950
test rows:         2,302

All checks passed.


## 4. Build Config and Train

In [8]:
random_seed = 1338
seed_everything(random_seed)
torch.set_default_device(device)

# ── Edit these ────────────────────────────────────────────────────────────────
MAX_EPOCHS = 1000
BATCH_SIZE = 256
SAVE_DIR   = f'{CKPT_DIR}/masksdm_butterfly_1339'
# ─────────────────────────────────────────────────────────────────────────────

config = {
    'device':                    device,
    'seed':                      random_seed,
    # 'dataset' is a SplotDataset/GeoPlantDataset selector in MaskSDM's
    # data_helpers.py:88 — keep 'splot' to use the generic SplotDataset wrapper.
    # This does NOT mean we are using sPlot data.
    'dataset':                   'splot',
    'n_features':                n_features,
    'n_species':                 n_species,
    'n_samples_train':           len(data['y_train']),
    'n_samples_val':             len(data['y_val']),
    'n_samples_test':            len(data['y_test']),
    'indices_evaluated_species': indices_evaluated,
    'n_evaluated_species':       len(indices_evaluated),
    # satclip=False: zeros are always fully masked so they carry no signal
    'satclip':                   False,
    'model':                     'FTTransformer',
    'd_hidden':                  192,
    'n_heads':                   8,
    'n_blocks':                  7,
    'dropout':                   0.1,
    'd_out':                     n_species,
    'epochs':                    MAX_EPOCHS,
    'batch_size':                BATCH_SIZE,
    'batch_size_eval':           4096,
    'loss':                      'weighted',
    'species_weights':           torch.tensor(
                                     len(data['y_train']) / (data['y_train'].sum(axis=0) + 1e-5),
                                     dtype=torch.float32
                                 ).to(device),
    'optimizer':                 'AdamW',
    'scheduler_free':            True,
    'lr':                        0.001,
    'weight_decay':              0.01,
    'warmup_steps':              1000,
    'masking':                   True,
    'extra_masking':             True,
    'save_dir':                  SAVE_DIR,
    'use_wandb':                 False,
    'wandb_init':                {},
}

print(f'n_features:  {n_features}')
print(f'n_species:   {n_species}')
print(f'max_epochs:  {MAX_EPOCHS}')
print(f'save_dir:    {SAVE_DIR}')
print(f'device:      {device}')

train(config, data)


n_features:  15
n_species:   171
max_epochs:  1000
save_dir:    /content/drive/MyDrive/CISO/eButterfly/masksdm_butterfly_1339
device:      cuda
Training model...
Epoch 1, val AUC: 0.520320
Epoch 2, val AUC: 0.556049
Epoch 3, val AUC: 0.570698
Epoch 4, val AUC: 0.582130
Epoch 5, val AUC: 0.605099
Epoch 6, val AUC: 0.660908
Epoch 7, val AUC: 0.715165
Epoch 8, val AUC: 0.728790
Epoch 9, val AUC: 0.732767
Epoch 10, val AUC: 0.736035
Epoch 11, val AUC: 0.745892
Epoch 12, val AUC: 0.750462
Epoch 13, val AUC: 0.758953
Epoch 14, val AUC: 0.766224
Epoch 15, val AUC: 0.769949
Epoch 16, val AUC: 0.768085
Epoch 17, val AUC: 0.770874
Epoch 18, val AUC: 0.775010
Epoch 19, val AUC: 0.776802
Epoch 20, val AUC: 0.782955
Epoch 21, val AUC: 0.767499
Epoch 22, val AUC: 0.780851
Epoch 23, val AUC: 0.789958
Epoch 24, val AUC: 0.797545
Epoch 25, val AUC: 0.801360
Epoch 26, val AUC: 0.803655
Epoch 27, val AUC: 0.806331
Epoch 28, val AUC: 0.808249
Epoch 29, val AUC: 0.808050
Epoch 30, val AUC: 0.807942
Epoch 3

FTTransformer(
  (feature_tokenizer): FeatureTokenizer(
    (periodic_embeddings): PeriodicEmbeddings(
      (periodic): _Periodic()
      (linear): _NLinear()
      (activation): ReLU()
    )
  )
  (satclip_projection): Linear(in_features=256, out_features=192, bias=True)
  (transformer_encoder): TransformerEncoder(
    (blocks): ModuleList(
      (0-6): 7 x ModuleDict(
        (attention_normalization): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
        (attention): MultiheadAttention(
          (W_q): Linear(in_features=192, out_features=192, bias=True)
          (W_k): Linear(in_features=192, out_features=192, bias=True)
          (W_v): Linear(in_features=192, out_features=192, bias=True)
          (W_out): Linear(in_features=192, out_features=192, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (attention_residual_dropout): Dropout(p=0.1, inplace=False)
        (ffn_normalization): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
  

In [9]:
import re

#BEST_EPOCH = S  # replace with the epoch number you found

#CHECKPOINT_PATH = f"{SAVE_DIR}/epoch_{BEST_EPOCH}.pt"

#print(f"Checkpoint: {CHECKPOINT_PATH}")


log = """Epoch 1, val AUC: 0.520320
Epoch 2, val AUC: 0.556049
Epoch 3, val AUC: 0.570698
Epoch 4, val AUC: 0.582130
Epoch 5, val AUC: 0.605099
Epoch 6, val AUC: 0.660908
Epoch 7, val AUC: 0.715165
Epoch 8, val AUC: 0.728790
Epoch 9, val AUC: 0.732767
Epoch 10, val AUC: 0.736035
Epoch 11, val AUC: 0.745892
Epoch 12, val AUC: 0.750462
Epoch 13, val AUC: 0.758953
Epoch 14, val AUC: 0.766224
Epoch 15, val AUC: 0.769949
Epoch 16, val AUC: 0.768085
Epoch 17, val AUC: 0.770874
Epoch 18, val AUC: 0.775010
Epoch 19, val AUC: 0.776802
Epoch 20, val AUC: 0.782955
Epoch 21, val AUC: 0.767499
Epoch 22, val AUC: 0.780851
Epoch 23, val AUC: 0.789958
Epoch 24, val AUC: 0.797545
Epoch 25, val AUC: 0.801360
Epoch 26, val AUC: 0.803655
Epoch 27, val AUC: 0.806331
Epoch 28, val AUC: 0.808249
Epoch 29, val AUC: 0.808050
Epoch 30, val AUC: 0.807942
Epoch 31, val AUC: 0.808143
Epoch 32, val AUC: 0.808755
Epoch 33, val AUC: 0.809258
Epoch 34, val AUC: 0.809913
Epoch 35, val AUC: 0.809676
Epoch 36, val AUC: 0.809404
Epoch 37, val AUC: 0.809334
Epoch 38, val AUC: 0.809438
Epoch 39, val AUC: 0.810032
Epoch 40, val AUC: 0.810432
Epoch 41, val AUC: 0.811707
Epoch 42, val AUC: 0.812645
Epoch 43, val AUC: 0.813446
Epoch 44, val AUC: 0.814709
Epoch 45, val AUC: 0.817127
Epoch 46, val AUC: 0.818411
Epoch 47, val AUC: 0.819364
Epoch 48, val AUC: 0.821063
Epoch 49, val AUC: 0.822197
Epoch 50, val AUC: 0.823353
Epoch 51, val AUC: 0.819699
Epoch 52, val AUC: 0.817962
Epoch 53, val AUC: 0.817347
Epoch 54, val AUC: 0.817541
Epoch 55, val AUC: 0.818424
Epoch 56, val AUC: 0.819589
Epoch 57, val AUC: 0.820847
Epoch 58, val AUC: 0.821830
Epoch 59, val AUC: 0.822682
Epoch 60, val AUC: 0.823726
Epoch 61, val AUC: 0.824785
Epoch 62, val AUC: 0.825669
Epoch 63, val AUC: 0.826799
Epoch 64, val AUC: 0.827552
Epoch 65, val AUC: 0.829094
Epoch 66, val AUC: 0.829903
Epoch 67, val AUC: 0.830649
Epoch 68, val AUC: 0.831252
Epoch 69, val AUC: 0.830298
Epoch 70, val AUC: 0.829760
Epoch 71, val AUC: 0.829346
Epoch 72, val AUC: 0.829475
Epoch 73, val AUC: 0.829711
Epoch 74, val AUC: 0.830202
Epoch 75, val AUC: 0.830812
Epoch 76, val AUC: 0.831128
Epoch 77, val AUC: 0.831459
Epoch 78, val AUC: 0.831853
Epoch 79, val AUC: 0.832428
Epoch 80, val AUC: 0.832860
Epoch 81, val AUC: 0.833327
Epoch 82, val AUC: 0.833769
Epoch 83, val AUC: 0.834153
Epoch 84, val AUC: 0.834814
Epoch 85, val AUC: 0.835327
Epoch 86, val AUC: 0.835824
Epoch 87, val AUC: 0.836185
Epoch 88, val AUC: 0.836498
Epoch 89, val AUC: 0.836913
Epoch 90, val AUC: 0.837435
Epoch 91, val AUC: 0.838078
Epoch 92, val AUC: 0.838658
Epoch 93, val AUC: 0.839343
Epoch 94, val AUC: 0.839749
Epoch 95, val AUC: 0.840177
Epoch 96, val AUC: 0.840762
Epoch 97, val AUC: 0.841163
Epoch 98, val AUC: 0.841213
Epoch 99, val AUC: 0.841377
Epoch 100, val AUC: 0.841670
Epoch 101, val AUC: 0.841651
Epoch 102, val AUC: 0.841316
Epoch 103, val AUC: 0.841273
Epoch 104, val AUC: 0.841288
Epoch 105, val AUC: 0.841401
Epoch 106, val AUC: 0.841392
Epoch 107, val AUC: 0.841312
Epoch 108, val AUC: 0.841361
Epoch 109, val AUC: 0.841594
Epoch 110, val AUC: 0.841890
Epoch 111, val AUC: 0.842092
Epoch 112, val AUC: 0.842412
Epoch 113, val AUC: 0.842884
Epoch 114, val AUC: 0.843708
Epoch 115, val AUC: 0.844001
Epoch 116, val AUC: 0.844381
Epoch 117, val AUC: 0.844600
Epoch 118, val AUC: 0.844663
Epoch 119, val AUC: 0.844759
Epoch 120, val AUC: 0.845038
Epoch 121, val AUC: 0.845095
Epoch 122, val AUC: 0.845305
Epoch 123, val AUC: 0.845454
Epoch 124, val AUC: 0.845662
Epoch 125, val AUC: 0.845941
Epoch 126, val AUC: 0.846106
Epoch 127, val AUC: 0.846259
Epoch 128, val AUC: 0.846396
Epoch 129, val AUC: 0.846599
Epoch 130, val AUC: 0.846937
Epoch 131, val AUC: 0.846849
Epoch 132, val AUC: 0.846749
Epoch 133, val AUC: 0.846773
Epoch 134, val AUC: 0.846714
Epoch 135, val AUC: 0.846721
Epoch 136, val AUC: 0.846723
Epoch 137, val AUC: 0.846759
Epoch 138, val AUC: 0.846820
Epoch 139, val AUC: 0.846669
Epoch 140, val AUC: 0.846686
Epoch 141, val AUC: 0.846695
Epoch 142, val AUC: 0.846631
Epoch 143, val AUC: 0.846682
Epoch 144, val AUC: 0.846940
Epoch 145, val AUC: 0.846895
Epoch 146, val AUC: 0.847020
Epoch 147, val AUC: 0.847087
Epoch 148, val AUC: 0.847118
Epoch 149, val AUC: 0.847141
Epoch 150, val AUC: 0.846847
Epoch 151, val AUC: 0.846474
Epoch 152, val AUC: 0.846193
Epoch 153, val AUC: 0.846140
Epoch 154, val AUC: 0.846229
Epoch 155, val AUC: 0.846294
Epoch 156, val AUC: 0.846441
Epoch 157, val AUC: 0.846403
Epoch 158, val AUC: 0.846532
Epoch 159, val AUC: 0.846589
Epoch 160, val AUC: 0.846631
Epoch 161, val AUC: 0.846608
Epoch 162, val AUC: 0.846624
Epoch 163, val AUC: 0.846684
Epoch 164, val AUC: 0.846669
Epoch 165, val AUC: 0.846684
Epoch 166, val AUC: 0.846682
Epoch 167, val AUC: 0.846852
Epoch 168, val AUC: 0.847055
Epoch 169, val AUC: 0.847218
Epoch 170, val AUC: 0.847345
Epoch 171, val AUC: 0.847177
Epoch 172, val AUC: 0.847087
Epoch 173, val AUC: 0.846956
Epoch 174, val AUC: 0.846850
Epoch 175, val AUC: 0.846875
Epoch 176, val AUC: 0.846914
Epoch 177, val AUC: 0.847061
Epoch 178, val AUC: 0.847245
Epoch 179, val AUC: 0.847402
Epoch 180, val AUC: 0.847659
Epoch 181, val AUC: 0.847790
Epoch 182, val AUC: 0.847865
Epoch 183, val AUC: 0.847822
Epoch 184, val AUC: 0.847821
Epoch 185, val AUC: 0.847903
Epoch 186, val AUC: 0.847917
Epoch 187, val AUC: 0.848044
Epoch 188, val AUC: 0.848080
Epoch 189, val AUC: 0.848196
Epoch 190, val AUC: 0.848252
Epoch 191, val AUC: 0.848259
Epoch 192, val AUC: 0.848348
Epoch 193, val AUC: 0.847719
Epoch 194, val AUC: 0.847321
Epoch 195, val AUC: 0.847156
Epoch 196, val AUC: 0.847069
Epoch 197, val AUC: 0.847005
Epoch 198, val AUC: 0.846873
Epoch 199, val AUC: 0.846776
Epoch 200, val AUC: 0.846632
Epoch 201, val AUC: 0.846594
Epoch 202, val AUC: 0.846467
Epoch 203, val AUC: 0.846323
Epoch 204, val AUC: 0.846315
Epoch 205, val AUC: 0.846270
Epoch 206, val AUC: 0.846342
Epoch 207, val AUC: 0.846338
Epoch 208, val AUC: 0.846408
Epoch 209, val AUC: 0.846325
Epoch 210, val AUC: 0.846181
Epoch 211, val AUC: 0.846074
Epoch 212, val AUC: 0.846044
Epoch 213, val AUC: 0.846103
Epoch 214, val AUC: 0.846169
Epoch 215, val AUC: 0.846330
Epoch 216, val AUC: 0.846454
Epoch 217, val AUC: 0.846576
Epoch 218, val AUC: 0.846669
Epoch 219, val AUC: 0.846771
Epoch 220, val AUC: 0.846808
Epoch 221, val AUC: 0.846720
Epoch 222, val AUC: 0.846687
Epoch 223, val AUC: 0.846623
Epoch 224, val AUC: 0.846471
Epoch 225, val AUC: 0.846389
Epoch 226, val AUC: 0.846020
Epoch 227, val AUC: 0.845634
Epoch 228, val AUC: 0.845379
Epoch 229, val AUC: 0.845247
Epoch 230, val AUC: 0.845132
Epoch 231, val AUC: 0.844978
Epoch 232, val AUC: 0.844848
Epoch 233, val AUC: 0.844684
Epoch 234, val AUC: 0.844600
Epoch 235, val AUC: 0.844392
Epoch 236, val AUC: 0.844228
Epoch 237, val AUC: 0.844062
Epoch 238, val AUC: 0.843651
Epoch 239, val AUC: 0.843336
Epoch 240, val AUC: 0.843152
Epoch 241, val AUC: 0.842991
Epoch 242, val AUC: 0.842898
Epoch 243, val AUC: 0.842779
Epoch 244, val AUC: 0.842575
Epoch 245, val AUC: 0.842477
Epoch 246, val AUC: 0.842426
Epoch 247, val AUC: 0.842510
Epoch 248, val AUC: 0.842497
Epoch 249, val AUC: 0.842491
Epoch 250, val AUC: 0.842651
Epoch 251, val AUC: 0.842699
Epoch 252, val AUC: 0.842791
Epoch 253, val AUC: 0.842820
Epoch 254, val AUC: 0.842788
Epoch 255, val AUC: 0.842716
Epoch 256, val AUC: 0.842621
Epoch 257, val AUC: 0.842555
Epoch 258, val AUC: 0.842557
Epoch 259, val AUC: 0.842488
Epoch 260, val AUC: 0.842338
Epoch 261, val AUC: 0.842166
Epoch 262, val AUC: 0.842155
Epoch 263, val AUC: 0.842177
Epoch 264, val AUC: 0.842127
Epoch 265, val AUC: 0.842090
Epoch 266, val AUC: 0.842010
Epoch 267, val AUC: 0.841951
Epoch 268, val AUC: 0.841958
Epoch 269, val AUC: 0.841918
Epoch 270, val AUC: 0.841852
Epoch 271, val AUC: 0.841840
Epoch 272, val AUC: 0.841762
Epoch 273, val AUC: 0.841707
Epoch 274, val AUC: 0.841662
Epoch 275, val AUC: 0.841698
Epoch 276, val AUC: 0.841656
Epoch 277, val AUC: 0.841688
Epoch 278, val AUC: 0.841788
Epoch 279, val AUC: 0.841851
Epoch 280, val AUC: 0.841838
Epoch 281, val AUC: 0.841865
Epoch 282, val AUC: 0.841814
Epoch 283, val AUC: 0.841704
Epoch 284, val AUC: 0.841575
Epoch 285, val AUC: 0.841579
Epoch 286, val AUC: 0.841615
Epoch 287, val AUC: 0.841537
Epoch 288, val AUC: 0.841453
Epoch 289, val AUC: 0.841406
Epoch 290, val AUC: 0.841369
Epoch 291, val AUC: 0.841257
Epoch 292, val AUC: 0.841182
Epoch 293, val AUC: 0.841115
Epoch 294, val AUC: 0.841122
Epoch 295, val AUC: 0.841122
Epoch 296, val AUC: 0.841052
Epoch 297, val AUC: 0.841137
Epoch 298, val AUC: 0.841152
Epoch 299, val AUC: 0.841233
Epoch 300, val AUC: 0.841234
Epoch 301, val AUC: 0.841202
Epoch 302, val AUC: 0.841146
Epoch 303, val AUC: 0.841076
Epoch 304, val AUC: 0.841048
Epoch 305, val AUC: 0.840967
Epoch 306, val AUC: 0.840960
Epoch 307, val AUC: 0.840797
Epoch 308, val AUC: 0.840696
Epoch 309, val AUC: 0.840677
Epoch 310, val AUC: 0.840600
Epoch 311, val AUC: 0.840556
Epoch 312, val AUC: 0.840558
Epoch 313, val AUC: 0.840496
Epoch 314, val AUC: 0.840510
Epoch 315, val AUC: 0.840520
Epoch 316, val AUC: 0.840542
Epoch 317, val AUC: 0.840530
Epoch 318, val AUC: 0.840566
Epoch 319, val AUC: 0.840623
Epoch 320, val AUC: 0.840610
Epoch 321, val AUC: 0.840666
Epoch 322, val AUC: 0.840785
Epoch 323, val AUC: 0.840930
Epoch 324, val AUC: 0.841068
Epoch 325, val AUC: 0.841117
Epoch 326, val AUC: 0.841163
Epoch 327, val AUC: 0.841259
Epoch 328, val AUC: 0.841308
Epoch 329, val AUC: 0.841360
Epoch 330, val AUC: 0.841461
Epoch 331, val AUC: 0.841633
Epoch 332, val AUC: 0.841695
Epoch 333, val AUC: 0.841736
Epoch 334, val AUC: 0.841790
Epoch 335, val AUC: 0.841864
Epoch 336, val AUC: 0.841937
Epoch 337, val AUC: 0.841939
Epoch 338, val AUC: 0.841989
Epoch 339, val AUC: 0.842091
Epoch 340, val AUC: 0.842110
Epoch 341, val AUC: 0.842255
Epoch 342, val AUC: 0.842308
Epoch 343, val AUC: 0.842333
Epoch 344, val AUC: 0.842377
Epoch 345, val AUC: 0.842402
Epoch 346, val AUC: 0.842476
Epoch 347, val AUC: 0.842479
Epoch 348, val AUC: 0.842518
Epoch 349, val AUC: 0.842445
Epoch 350, val AUC: 0.842513
Epoch 351, val AUC: 0.842673
Epoch 352, val AUC: 0.842815
Epoch 353, val AUC: 0.842832
Epoch 354, val AUC: 0.842817
Epoch 355, val AUC: 0.842821
Epoch 356, val AUC: 0.842909
Epoch 357, val AUC: 0.842925
Epoch 358, val AUC: 0.842907
Epoch 359, val AUC: 0.843000
Epoch 360, val AUC: 0.843107
Epoch 361, val AUC: 0.843186
Epoch 362, val AUC: 0.843267
Epoch 363, val AUC: 0.843338
Epoch 364, val AUC: 0.843388
Epoch 365, val AUC: 0.843472
Epoch 366, val AUC: 0.843455
Epoch 367, val AUC: 0.843500
Epoch 368, val AUC: 0.843512
Epoch 369, val AUC: 0.843549
Epoch 370, val AUC: 0.843500
Epoch 371, val AUC: 0.843518
Epoch 372, val AUC: 0.843631
Epoch 373, val AUC: 0.843854
Epoch 374, val AUC: 0.844005
Epoch 375, val AUC: 0.844198
Epoch 376, val AUC: 0.844316
Epoch 377, val AUC: 0.844372
Epoch 378, val AUC: 0.844479
Epoch 379, val AUC: 0.844544
Epoch 380, val AUC: 0.844602
Epoch 381, val AUC: 0.844650
Epoch 382, val AUC: 0.844725
Epoch 383, val AUC: 0.844732
Epoch 384, val AUC: 0.844807
Epoch 385, val AUC: 0.844887
Epoch 386, val AUC: 0.844938
Epoch 387, val AUC: 0.845025
Epoch 388, val AUC: 0.845066
Epoch 389, val AUC: 0.845121
Epoch 390, val AUC: 0.845151
Epoch 391, val AUC: 0.845199
Epoch 392, val AUC: 0.845175
Epoch 393, val AUC: 0.845221
Epoch 394, val AUC: 0.845285
Epoch 395, val AUC: 0.845469
Epoch 396, val AUC: 0.845542
Epoch 397, val AUC: 0.845670
Epoch 398, val AUC: 0.845786
Epoch 399, val AUC: 0.845829
Epoch 400, val AUC: 0.845889
Epoch 401, val AUC: 0.845981
Epoch 402, val AUC: 0.846032
Epoch 403, val AUC: 0.846102
Epoch 404, val AUC: 0.846093
Epoch 405, val AUC: 0.846173
Epoch 406, val AUC: 0.846160
Epoch 407, val AUC: 0.846146
Epoch 408, val AUC: 0.846161
Epoch 409, val AUC: 0.846348
Epoch 410, val AUC: 0.846446
Epoch 411, val AUC: 0.846585
Epoch 412, val AUC: 0.846624
Epoch 413, val AUC: 0.846689
Epoch 414, val AUC: 0.846714
Epoch 415, val AUC: 0.846744
Epoch 416, val AUC: 0.846795
Epoch 417, val AUC: 0.846933
Epoch 418, val AUC: 0.846933
Epoch 419, val AUC: 0.846941
Epoch 420, val AUC: 0.846954
Epoch 421, val AUC: 0.846971
Epoch 422, val AUC: 0.846932
Epoch 423, val AUC: 0.846900
Epoch 424, val AUC: 0.846884
Epoch 425, val AUC: 0.846929
Epoch 426, val AUC: 0.846919
Epoch 427, val AUC: 0.846985
Epoch 428, val AUC: 0.846933
Epoch 429, val AUC: 0.846884
Epoch 430, val AUC: 0.846864
Epoch 431, val AUC: 0.846781
Epoch 432, val AUC: 0.846738
Epoch 433, val AUC: 0.846670
Epoch 434, val AUC: 0.846649
Epoch 435, val AUC: 0.846662
Epoch 436, val AUC: 0.846717
Epoch 437, val AUC: 0.846737
Epoch 438, val AUC: 0.846764
Epoch 439, val AUC: 0.846830
Epoch 440, val AUC: 0.846785
Epoch 441, val AUC: 0.846822
Epoch 442, val AUC: 0.846857
Epoch 443, val AUC: 0.846856
Epoch 444, val AUC: 0.846860
Epoch 445, val AUC: 0.846866
Epoch 446, val AUC: 0.846890
Epoch 447, val AUC: 0.846873
Epoch 448, val AUC: 0.846868
Epoch 449, val AUC: 0.846959
Epoch 450, val AUC: 0.847056
Epoch 451, val AUC: 0.847122
Epoch 452, val AUC: 0.847181
Epoch 453, val AUC: 0.847190
Epoch 454, val AUC: 0.847238
Epoch 455, val AUC: 0.847202
Epoch 456, val AUC: 0.847197
Epoch 457, val AUC: 0.847206
Epoch 458, val AUC: 0.847243
Epoch 459, val AUC: 0.847308
Epoch 460, val AUC: 0.847365
Epoch 461, val AUC: 0.847412
Epoch 462, val AUC: 0.847474
Epoch 463, val AUC: 0.847513
Epoch 464, val AUC: 0.847554
Epoch 465, val AUC: 0.847588
Epoch 466, val AUC: 0.847658
Epoch 467, val AUC: 0.847729
Epoch 468, val AUC: 0.847772
Epoch 469, val AUC: 0.847836
Epoch 470, val AUC: 0.847873
Epoch 471, val AUC: 0.847800
Epoch 472, val AUC: 0.847764
Epoch 473, val AUC: 0.847961
Epoch 474, val AUC: 0.847990
Epoch 475, val AUC: 0.848059
Epoch 476, val AUC: 0.848116
Epoch 477, val AUC: 0.848156
Epoch 478, val AUC: 0.848187
Epoch 479, val AUC: 0.848297
Epoch 480, val AUC: 0.848380
Epoch 481, val AUC: 0.848467
Epoch 482, val AUC: 0.848531
Epoch 483, val AUC: 0.848563
Epoch 484, val AUC: 0.848506
Epoch 485, val AUC: 0.848602
Epoch 486, val AUC: 0.848610
Epoch 487, val AUC: 0.848681
Epoch 488, val AUC: 0.848699
Epoch 489, val AUC: 0.848689
Epoch 490, val AUC: 0.848723
Epoch 491, val AUC: 0.848743
Epoch 492, val AUC: 0.848816
Epoch 493, val AUC: 0.848833
Epoch 494, val AUC: 0.848812
Epoch 495, val AUC: 0.848788
Epoch 496, val AUC: 0.848774
Epoch 497, val AUC: 0.848765
Epoch 498, val AUC: 0.848747
Epoch 499, val AUC: 0.848679
Epoch 500, val AUC: 0.848654
Epoch 501, val AUC: 0.848679
Epoch 502, val AUC: 0.848644
Epoch 503, val AUC: 0.848599
Epoch 504, val AUC: 0.848623
Epoch 505, val AUC: 0.848505
Epoch 506, val AUC: 0.848408
Epoch 507, val AUC: 0.848400
Epoch 508, val AUC: 0.848392
Epoch 509, val AUC: 0.848387
Epoch 510, val AUC: 0.848388
Epoch 511, val AUC: 0.848300
Epoch 512, val AUC: 0.848239
Epoch 513, val AUC: 0.848178
Epoch 514, val AUC: 0.848036
Epoch 515, val AUC: 0.847935
Epoch 516, val AUC: 0.847860
Epoch 517, val AUC: 0.847800
Epoch 518, val AUC: 0.847763
Epoch 519, val AUC: 0.847727
Epoch 520, val AUC: 0.847648
Epoch 521, val AUC: 0.847570
Epoch 522, val AUC: 0.847542
Epoch 523, val AUC: 0.847461
Epoch 524, val AUC: 0.847463
Epoch 525, val AUC: 0.847367
Epoch 526, val AUC: 0.847275
Epoch 527, val AUC: 0.847234
Epoch 528, val AUC: 0.847193
Epoch 529, val AUC: 0.847185
Epoch 530, val AUC: 0.847157
Epoch 531, val AUC: 0.847056
Epoch 532, val AUC: 0.846998
Epoch 533, val AUC: 0.846987
Epoch 534, val AUC: 0.846929
Epoch 535, val AUC: 0.846859
Epoch 536, val AUC: 0.846809
Epoch 537, val AUC: 0.846754
Epoch 538, val AUC: 0.846658
Epoch 539, val AUC: 0.846542
Epoch 540, val AUC: 0.846532
Epoch 541, val AUC: 0.846372
Epoch 542, val AUC: 0.846301
Epoch 543, val AUC: 0.846161
Epoch 544, val AUC: 0.846139
Epoch 545, val AUC: 0.846071
Epoch 546, val AUC: 0.846021
Epoch 547, val AUC: 0.845988
Epoch 548, val AUC: 0.845989
Epoch 549, val AUC: 0.845972
Epoch 550, val AUC: 0.845961
Epoch 551, val AUC: 0.845925
Epoch 552, val AUC: 0.845954
Epoch 553, val AUC: 0.845973
Epoch 554, val AUC: 0.846026
Epoch 555, val AUC: 0.846058
Epoch 556, val AUC: 0.846059
Epoch 557, val AUC: 0.846052
Epoch 558, val AUC: 0.846049
Epoch 559, val AUC: 0.846008
Epoch 560, val AUC: 0.846027
Epoch 561, val AUC: 0.845997
Epoch 562, val AUC: 0.846059
Epoch 563, val AUC: 0.846075
Epoch 564, val AUC: 0.846080
Epoch 565, val AUC: 0.846115
Epoch 566, val AUC: 0.846182
Epoch 567, val AUC: 0.846172
Epoch 568, val AUC: 0.846229
Epoch 569, val AUC: 0.846202
Epoch 570, val AUC: 0.846207
Epoch 571, val AUC: 0.846221
Epoch 572, val AUC: 0.846242
Epoch 573, val AUC: 0.846255
Epoch 574, val AUC: 0.846273
Epoch 575, val AUC: 0.846322
Epoch 576, val AUC: 0.846289
Epoch 577, val AUC: 0.846224
Epoch 578, val AUC: 0.846200
Epoch 579, val AUC: 0.846139
Epoch 580, val AUC: 0.846157
Epoch 581, val AUC: 0.846132
Epoch 582, val AUC: 0.846070
Epoch 583, val AUC: 0.846083
Epoch 584, val AUC: 0.845987
Epoch 585, val AUC: 0.845893
Epoch 586, val AUC: 0.845872
Epoch 587, val AUC: 0.845790
Epoch 588, val AUC: 0.845746
Epoch 589, val AUC: 0.845632
Epoch 590, val AUC: 0.845530
Epoch 591, val AUC: 0.845478
Epoch 592, val AUC: 0.845395
Epoch 593, val AUC: 0.845369
Epoch 594, val AUC: 0.845332
Epoch 595, val AUC: 0.845324
Epoch 596, val AUC: 0.845271
Epoch 597, val AUC: 0.845208
Epoch 598, val AUC: 0.845220
Epoch 599, val AUC: 0.845210
Epoch 600, val AUC: 0.845153
Epoch 601, val AUC: 0.845080
Epoch 602, val AUC: 0.845028
Epoch 603, val AUC: 0.844970
Epoch 604, val AUC: 0.844946
Epoch 605, val AUC: 0.844879
Epoch 606, val AUC: 0.844744
Epoch 607, val AUC: 0.844708
Epoch 608, val AUC: 0.844759
Epoch 609, val AUC: 0.844763
Epoch 610, val AUC: 0.844710
Epoch 611, val AUC: 0.844682
Epoch 612, val AUC: 0.844643
Epoch 613, val AUC: 0.844593
Epoch 614, val AUC: 0.844532
Epoch 615, val AUC: 0.844508
Epoch 616, val AUC: 0.844443
Epoch 617, val AUC: 0.844376
Epoch 618, val AUC: 0.844283
Epoch 619, val AUC: 0.844251
Epoch 620, val AUC: 0.844194
Epoch 621, val AUC: 0.844120
Epoch 622, val AUC: 0.844087
Epoch 623, val AUC: 0.844043
Epoch 624, val AUC: 0.843975
Epoch 625, val AUC: 0.843896
Epoch 626, val AUC: 0.843914
Epoch 627, val AUC: 0.843898
Epoch 628, val AUC: 0.843831
Epoch 629, val AUC: 0.843804
Epoch 630, val AUC: 0.843785
Epoch 631, val AUC: 0.843726
Epoch 632, val AUC: 0.843678
Epoch 633, val AUC: 0.843565
Epoch 634, val AUC: 0.843552
Epoch 635, val AUC: 0.843545
Epoch 636, val AUC: 0.843550
Epoch 637, val AUC: 0.843536
Epoch 638, val AUC: 0.843513
Epoch 639, val AUC: 0.843516
Epoch 640, val AUC: 0.843501
Epoch 641, val AUC: 0.843498
Epoch 642, val AUC: 0.843426
Epoch 643, val AUC: 0.843409
Epoch 644, val AUC: 0.843418
Epoch 645, val AUC: 0.843361
Epoch 646, val AUC: 0.843304
Epoch 647, val AUC: 0.843283
Epoch 648, val AUC: 0.843242
Epoch 649, val AUC: 0.843179
Epoch 650, val AUC: 0.843100
Epoch 651, val AUC: 0.843054
Epoch 652, val AUC: 0.843025
Epoch 653, val AUC: 0.842959
Epoch 654, val AUC: 0.842959
Epoch 655, val AUC: 0.842889
Epoch 656, val AUC: 0.842909
Epoch 657, val AUC: 0.842830
Epoch 658, val AUC: 0.842781
Epoch 659, val AUC: 0.842728
Epoch 660, val AUC: 0.842746
Epoch 661, val AUC: 0.842767
Epoch 662, val AUC: 0.842763
Epoch 663, val AUC: 0.842751
Epoch 664, val AUC: 0.842798
Epoch 665, val AUC: 0.842803
Epoch 666, val AUC: 0.842835
Epoch 667, val AUC: 0.842886
Epoch 668, val AUC: 0.842893
Epoch 669, val AUC: 0.842878
Epoch 670, val AUC: 0.842898
Epoch 671, val AUC: 0.842982
Epoch 672, val AUC: 0.843013
Epoch 673, val AUC: 0.843024
Epoch 674, val AUC: 0.843023
Epoch 675, val AUC: 0.842997
Epoch 676, val AUC: 0.843059
Epoch 677, val AUC: 0.843073
Epoch 678, val AUC: 0.843077
Epoch 679, val AUC: 0.843179
Epoch 680, val AUC: 0.843206
Epoch 681, val AUC: 0.843257
Epoch 682, val AUC: 0.843241
Epoch 683, val AUC: 0.843247
Epoch 684, val AUC: 0.843276
Epoch 685, val AUC: 0.843281
Epoch 686, val AUC: 0.843290
Epoch 687, val AUC: 0.843238
Epoch 688, val AUC: 0.843244
Epoch 689, val AUC: 0.843269
Epoch 690, val AUC: 0.843266
Epoch 691, val AUC: 0.843273
Epoch 692, val AUC: 0.843330
Epoch 693, val AUC: 0.843344
Epoch 694, val AUC: 0.843359
Epoch 695, val AUC: 0.843399
Epoch 696, val AUC: 0.843405
Epoch 697, val AUC: 0.843376
Epoch 698, val AUC: 0.843374
Epoch 699, val AUC: 0.843407
Epoch 700, val AUC: 0.843387
Epoch 701, val AUC: 0.843426
Epoch 702, val AUC: 0.843438
Epoch 703, val AUC: 0.843457
Epoch 704, val AUC: 0.843480
Epoch 705, val AUC: 0.843540
Epoch 706, val AUC: 0.843528
Epoch 707, val AUC: 0.843569
Epoch 708, val AUC: 0.843604
Epoch 709, val AUC: 0.843603
Epoch 710, val AUC: 0.843593
Epoch 711, val AUC: 0.843564
Epoch 712, val AUC: 0.843529
Epoch 713, val AUC: 0.843562
Epoch 714, val AUC: 0.843513
Epoch 715, val AUC: 0.843445
Epoch 716, val AUC: 0.843412
Epoch 717, val AUC: 0.843373
Epoch 718, val AUC: 0.843321
Epoch 719, val AUC: 0.843329
Epoch 720, val AUC: 0.843290
Epoch 721, val AUC: 0.843320
Epoch 722, val AUC: 0.843289
Epoch 723, val AUC: 0.843307
Epoch 724, val AUC: 0.843301
Epoch 725, val AUC: 0.843226
Epoch 726, val AUC: 0.843210
Epoch 727, val AUC: 0.843181
Epoch 728, val AUC: 0.843136
Epoch 729, val AUC: 0.843099
Epoch 730, val AUC: 0.843123
Epoch 731, val AUC: 0.843080
Epoch 732, val AUC: 0.843075
Epoch 733, val AUC: 0.843095
Epoch 734, val AUC: 0.843109
Epoch 735, val AUC: 0.843125
Epoch 736, val AUC: 0.843154
Epoch 737, val AUC: 0.843131
Epoch 738, val AUC: 0.843094
Epoch 739, val AUC: 0.843114
Epoch 740, val AUC: 0.843159
Epoch 741, val AUC: 0.843109
Epoch 742, val AUC: 0.843082
Epoch 743, val AUC: 0.843050
Epoch 744, val AUC: 0.843054
Epoch 745, val AUC: 0.843061
Epoch 746, val AUC: 0.842994
Epoch 747, val AUC: 0.842996
Epoch 748, val AUC: 0.842988
Epoch 749, val AUC: 0.842982
Epoch 750, val AUC: 0.842998
Epoch 751, val AUC: 0.843014
Epoch 752, val AUC: 0.842990
Epoch 753, val AUC: 0.843011
Epoch 754, val AUC: 0.842993
Epoch 755, val AUC: 0.843002
Epoch 756, val AUC: 0.842999
Epoch 757, val AUC: 0.842993
Epoch 758, val AUC: 0.843049
Epoch 759, val AUC: 0.842994
Epoch 760, val AUC: 0.843010
Epoch 761, val AUC: 0.842989
Epoch 762, val AUC: 0.843043
Epoch 763, val AUC: 0.843049
Epoch 764, val AUC: 0.843077
Epoch 765, val AUC: 0.843103
Epoch 766, val AUC: 0.843113
Epoch 767, val AUC: 0.843159
Epoch 768, val AUC: 0.843187
Epoch 769, val AUC: 0.843176
Epoch 770, val AUC: 0.843214
Epoch 771, val AUC: 0.843231
Epoch 772, val AUC: 0.843274
Epoch 773, val AUC: 0.843328
Epoch 774, val AUC: 0.843335
Epoch 775, val AUC: 0.843370
Epoch 776, val AUC: 0.843417
Epoch 777, val AUC: 0.843512
Epoch 778, val AUC: 0.843521
Epoch 779, val AUC: 0.843584
Epoch 780, val AUC: 0.843694
Epoch 781, val AUC: 0.843723
Epoch 782, val AUC: 0.843757
Epoch 783, val AUC: 0.843821
Epoch 784, val AUC: 0.843853
Epoch 785, val AUC: 0.843882
Epoch 786, val AUC: 0.843905
Epoch 787, val AUC: 0.843969
Epoch 788, val AUC: 0.843965
Epoch 789, val AUC: 0.844000
Epoch 790, val AUC: 0.844035
Epoch 791, val AUC: 0.844049
Epoch 792, val AUC: 0.844085
Epoch 793, val AUC: 0.844134
Epoch 794, val AUC: 0.844191
Epoch 795, val AUC: 0.844210
Epoch 796, val AUC: 0.844220
Epoch 797, val AUC: 0.844285
Epoch 798, val AUC: 0.844318
Epoch 799, val AUC: 0.844329
Epoch 800, val AUC: 0.844397
Epoch 801, val AUC: 0.844413
Epoch 802, val AUC: 0.844512
Epoch 803, val AUC: 0.844560
Epoch 804, val AUC: 0.844586
Epoch 805, val AUC: 0.844671
Epoch 806, val AUC: 0.844704
Epoch 807, val AUC: 0.844736
Epoch 808, val AUC: 0.844780
Epoch 809, val AUC: 0.844858
Epoch 810, val AUC: 0.844874
Epoch 811, val AUC: 0.844901
Epoch 812, val AUC: 0.844927
Epoch 813, val AUC: 0.844943
Epoch 814, val AUC: 0.844980
Epoch 815, val AUC: 0.844973
Epoch 816, val AUC: 0.845015
Epoch 817, val AUC: 0.845038
Epoch 818, val AUC: 0.845084
Epoch 819, val AUC: 0.845097
Epoch 820, val AUC: 0.845103
Epoch 821, val AUC: 0.845086
Epoch 822, val AUC: 0.845095
Epoch 823, val AUC: 0.845091
Epoch 824, val AUC: 0.845115
Epoch 825, val AUC: 0.845136
Epoch 826, val AUC: 0.845148
Epoch 827, val AUC: 0.845133
Epoch 828, val AUC: 0.845127
Epoch 829, val AUC: 0.845194
Epoch 830, val AUC: 0.845214
Epoch 831, val AUC: 0.845178
Epoch 832, val AUC: 0.845195
Epoch 833, val AUC: 0.845222
Epoch 834, val AUC: 0.845252
Epoch 835, val AUC: 0.845286
Epoch 836, val AUC: 0.845318
Epoch 837, val AUC: 0.845318
Epoch 838, val AUC: 0.845319
Epoch 839, val AUC: 0.845359
Epoch 840, val AUC: 0.845380
Epoch 841, val AUC: 0.845386
Epoch 842, val AUC: 0.845389
Epoch 843, val AUC: 0.845301
Epoch 844, val AUC: 0.845234
Epoch 845, val AUC: 0.845189
Epoch 846, val AUC: 0.845077
Epoch 847, val AUC: 0.845072
Epoch 848, val AUC: 0.845023
Epoch 849, val AUC: 0.845008
Epoch 850, val AUC: 0.844983
Epoch 851, val AUC: 0.844962
Epoch 852, val AUC: 0.844943
Epoch 853, val AUC: 0.844937
Epoch 854, val AUC: 0.844897
Epoch 855, val AUC: 0.844871
Epoch 856, val AUC: 0.844875
Epoch 857, val AUC: 0.844833
Epoch 858, val AUC: 0.844800
Epoch 859, val AUC: 0.844834
Epoch 860, val AUC: 0.844820
Epoch 861, val AUC: 0.844821
Epoch 862, val AUC: 0.844806
Epoch 863, val AUC: 0.844777
Epoch 864, val AUC: 0.844810
Epoch 865, val AUC: 0.844777
Epoch 866, val AUC: 0.844767
Epoch 867, val AUC: 0.844741
Epoch 868, val AUC: 0.844691
Epoch 869, val AUC: 0.844650
Epoch 870, val AUC: 0.844623
Epoch 871, val AUC: 0.844602
Epoch 872, val AUC: 0.844612
Epoch 873, val AUC: 0.844563
Epoch 874, val AUC: 0.844509
Epoch 875, val AUC: 0.844550
Epoch 876, val AUC: 0.844487
Epoch 877, val AUC: 0.844474
Epoch 878, val AUC: 0.844419
Epoch 879, val AUC: 0.844351
Epoch 880, val AUC: 0.844307
Epoch 881, val AUC: 0.844244
Epoch 882, val AUC: 0.844238
Epoch 883, val AUC: 0.844204
Epoch 884, val AUC: 0.844180
Epoch 885, val AUC: 0.844158
Epoch 886, val AUC: 0.844105
Epoch 887, val AUC: 0.844106
Epoch 888, val AUC: 0.844051
Epoch 889, val AUC: 0.843955
Epoch 890, val AUC: 0.843890
Epoch 891, val AUC: 0.843862
Epoch 892, val AUC: 0.843831
Epoch 893, val AUC: 0.843828
Epoch 894, val AUC: 0.843795
Epoch 895, val AUC: 0.843765
Epoch 896, val AUC: 0.843750
Epoch 897, val AUC: 0.843715
Epoch 898, val AUC: 0.843701
Epoch 899, val AUC: 0.843652
Epoch 900, val AUC: 0.843598
Epoch 901, val AUC: 0.843562
Epoch 902, val AUC: 0.843543
Epoch 903, val AUC: 0.843492
Epoch 904, val AUC: 0.843462
Epoch 905, val AUC: 0.843425
Epoch 906, val AUC: 0.843391
Epoch 907, val AUC: 0.843340
Epoch 908, val AUC: 0.843299
Epoch 909, val AUC: 0.843272
Epoch 910, val AUC: 0.843265
Epoch 911, val AUC: 0.843232
Epoch 912, val AUC: 0.843229
Epoch 913, val AUC: 0.843210
Epoch 914, val AUC: 0.843211
Epoch 915, val AUC: 0.843197
Epoch 916, val AUC: 0.843156
Epoch 917, val AUC: 0.843125
Epoch 918, val AUC: 0.843129
Epoch 919, val AUC: 0.843077
Epoch 920, val AUC: 0.843027
Epoch 921, val AUC: 0.842967
Epoch 922, val AUC: 0.842949
Epoch 923, val AUC: 0.842915
Epoch 924, val AUC: 0.842888
Epoch 925, val AUC: 0.842848
Epoch 926, val AUC: 0.842845
Epoch 927, val AUC: 0.842831
Epoch 928, val AUC: 0.842818
Epoch 929, val AUC: 0.842836
Epoch 930, val AUC: 0.842789
Epoch 931, val AUC: 0.842796
Epoch 932, val AUC: 0.842794
Epoch 933, val AUC: 0.842776
Epoch 934, val AUC: 0.842740
Epoch 935, val AUC: 0.842724
Epoch 936, val AUC: 0.842716
Epoch 937, val AUC: 0.842714
Epoch 938, val AUC: 0.842709
Epoch 939, val AUC: 0.842711
Epoch 940, val AUC: 0.842685
Epoch 941, val AUC: 0.842646
Epoch 942, val AUC: 0.842648
Epoch 943, val AUC: 0.842697
Epoch 944, val AUC: 0.842674
Epoch 945, val AUC: 0.842679
Epoch 946, val AUC: 0.842659
Epoch 947, val AUC: 0.842629
Epoch 948, val AUC: 0.842606
Epoch 949, val AUC: 0.842606
Epoch 950, val AUC: 0.842642
Epoch 951, val AUC: 0.842630
Epoch 952, val AUC: 0.842619
Epoch 953, val AUC: 0.842617
Epoch 954, val AUC: 0.842610
Epoch 955, val AUC: 0.842598
Epoch 956, val AUC: 0.842595
Epoch 957, val AUC: 0.842619
Epoch 958, val AUC: 0.842592
Epoch 959, val AUC: 0.842539
Epoch 960, val AUC: 0.842534
Epoch 961, val AUC: 0.842467
Epoch 962, val AUC: 0.842472
Epoch 963, val AUC: 0.842452
Epoch 964, val AUC: 0.842411
Epoch 965, val AUC: 0.842383
Epoch 966, val AUC: 0.842372
Epoch 967, val AUC: 0.842365
Epoch 968, val AUC: 0.842380
Epoch 969, val AUC: 0.842346
Epoch 970, val AUC: 0.842395
Epoch 971, val AUC: 0.842423
Epoch 972, val AUC: 0.842419
Epoch 973, val AUC: 0.842385
Epoch 974, val AUC: 0.842366
Epoch 975, val AUC: 0.842384
Epoch 976, val AUC: 0.842397
Epoch 977, val AUC: 0.842417
Epoch 978, val AUC: 0.842417
Epoch 979, val AUC: 0.842449
Epoch 980, val AUC: 0.842483
Epoch 981, val AUC: 0.842517
Epoch 982, val AUC: 0.842502
Epoch 983, val AUC: 0.842503
Epoch 984, val AUC: 0.842476
Epoch 985, val AUC: 0.842452
Epoch 986, val AUC: 0.842464
Epoch 987, val AUC: 0.842493
Epoch 988, val AUC: 0.842516
Epoch 989, val AUC: 0.842499
Epoch 990, val AUC: 0.842492
Epoch 991, val AUC: 0.842490
Epoch 992, val AUC: 0.842457
Epoch 993, val AUC: 0.842442
Epoch 994, val AUC: 0.842445
Epoch 995, val AUC: 0.842401
Epoch 996, val AUC: 0.842364
Epoch 997, val AUC: 0.842348
Epoch 998, val AUC: 0.842302
Epoch 999, val AUC: 0.842278
Epoch 1000, val AUC: 0.842264"""

matches = [
    (int(m.group(1)), float(m.group(2)))
    for m in re.finditer(r"Epoch (\d+), val AUC: ([\d.]+)", log)
]

if not matches:
    raise ValueError("No lines matching 'Epoch <n>, val AUC: <value>' were found in log.")

epoch, auc = max(matches, key=lambda x: x[1])

CHECKPOINT_PATH = f"{SAVE_DIR}/epoch_{epoch}.pt"

print(f"Best epoch: {epoch}  (val AUC = {auc:.6f})")
print(f"Checkpoint: {CHECKPOINT_PATH}")

Best epoch: 493  (val AUC = 0.848833)
Checkpoint: /content/drive/MyDrive/CISO/eButterfly/masksdm_butterfly_1339/epoch_493.pt


## 5. Inference with 100% Masking (p=1.0)

All tabular predictors are masked — the model receives only the learned mask token for every input.
This matches STEM-LM p=1.0 (fully unconditioned).
MaskSDM is trained with masked data modelling so this is the setting it is designed to handle gracefully.

In [10]:
model = get_model(config).to(device)
state_dict = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(state_dict)
model.eval()
print('Model loaded.')

Model loaded.


In [14]:
test_loader = DataLoader(
    get_torch_dataset(config, data['x_test'], data['y_test'], data['satclip_embeddings_test']),
    batch_size=config['batch_size_eval'],
    shuffle=False,
)

all_probs, all_targets = [], []

with torch.no_grad():
    for batch in test_loader:
        x_batch, y_batch, satclip_emb = batch
        x_batch     = x_batch.to(device)
        satclip_emb = satclip_emb.to(device)

        # Match upstream MaskSDM evaluate(): keep all valid features (mask=1=kept).
        # SatCLIP stays masked because we did not train with SatCLIP embeddings.
        x_mask       = ~torch.isnan(x_batch).to(device)
        satclip_mask = torch.zeros(len(satclip_emb), dtype=torch.bool, device=device)

        logits = model(x_batch, satclip_emb, x_mask, satclip_mask)
        probs  = torch.sigmoid(logits)

        all_probs.append(probs.cpu().numpy())
        all_targets.append(y_batch.cpu().numpy())

probs   = np.concatenate(all_probs,   axis=0)                    # (N_test, S)
targets = np.concatenate(all_targets, axis=0).astype(np.int64)   # (N_test, S)

print(f'probs:   {probs.shape}')
print(f'targets: {targets.shape}')

probs:   (2302, 171)
targets: (2302, 171)


## 6. STEM-LM Metrics

In [11]:
# ── Identical metric primitives to CISO notebook (STEMLM_metric.py) ──────────

def _safe_auc_roc(y, p):
    if y.size == 0 or len(set(y.tolist())) < 2 or np.isnan(p).any(): return float('nan')
    try: return float(roc_auc_score(y, p))
    except Exception: return float('nan')

def _safe_auc_pr(y, p):
    if y.size == 0 or y.sum() == 0 or y.sum() == y.size or np.isnan(p).any(): return float('nan')
    try: return float(average_precision_score(y, p))
    except Exception: return float('nan')

def _safe_brier(y, p):
    if y.size == 0 or np.isnan(p).any(): return float('nan')
    return float(np.mean((p - y.astype(np.float64))**2))

def _safe_ece(y, p, n_bins=15):
    if y.size == 0 or np.isnan(p).any(): return float('nan')
    edges = np.linspace(0, 1, n_bins+1)
    idx   = np.clip(np.digitize(p, edges) - 1, 0, n_bins-1)
    err = 0.0; n = p.size
    for b in range(n_bins):
        m = idx == b
        if not m.any(): continue
        err += (m.sum()/n) * abs(y[m].mean() - p[m].mean())
    return float(err)

def _safe_cbi(y, p, n_windows=101, width=0.1, min_per_window=10):
    if y.size == 0 or y.sum() == 0 or y.sum() == y.size or np.isnan(p).any(): return float('nan')
    pres = p[y==1]; bg = p[y==0]
    if pres.size == 0 or bg.size == 0: return float('nan')
    centers = np.linspace(0, 1, n_windows); half = width/2
    pe = np.full(n_windows, np.nan)
    for i, c in enumerate(centers):
        lo, hi = c-half, c+half
        n_bg = int(((bg>=lo)&(bg<=hi)).sum())
        if n_bg < min_per_window: continue
        e = n_bg/bg.size
        if e == 0: continue
        pe[i] = ((pres>=lo)&(pres<=hi)).sum()/pres.size / e
    ok = np.isfinite(pe)
    if ok.sum() < 3 or np.unique(pe[ok]).size < 2: return float('nan')
    try:
        rho = spearmanr(centers[ok], pe[ok]).statistic
        return float(rho) if np.isfinite(rho) else float('nan')
    except Exception: return float('nan')

print('Metric functions defined.')

Metric functions defined.


In [15]:
# At p=1.0 every feature is masked for every row, so we use all test rows.
# Per-split species filter: skip species with no presences (or no absences) in test
# — matches how the R baselines (Logistic / GAM / Maxnet) report n_species.
auc_roc_vals, auc_pr_vals, cbi_vals, brier_vals, ece_vals = [], [], [], [], []

for s in range(targets.shape[1]):
    y = targets[:, s].astype(np.int64)
    p = probs[:, s].astype(np.float64)
    if y.sum() == 0 or y.sum() == len(y):
        continue
    auc_roc_vals.append(_safe_auc_roc(y, p))
    auc_pr_vals.append(_safe_auc_pr(y, p))
    cbi_vals.append(_safe_cbi(y, p))
    brier_vals.append(_safe_brier(y, p))
    ece_vals.append(_safe_ece(y, p))

def _nanmean(v):   return float(np.nanmean([x for x in v if np.isfinite(x)])) if v else float('nan')
def _nanq(v, q):   vals=[x for x in v if np.isfinite(x)]; return float(np.quantile(vals,q)) if vals else float('nan')

summary = {
    'model':          'MaskSDM',
    'masking_p':      1.0,
    'eval_known_ratio': 0.0,
    'n_species':      len(auc_roc_vals),
    'mean_auc_roc':   _nanmean(auc_roc_vals),
    'auc_roc_q25':    _nanq(auc_roc_vals, 0.25),
    'auc_roc_q50':    _nanq(auc_roc_vals, 0.50),
    'auc_roc_q75':    _nanq(auc_roc_vals, 0.75),
    'mean_auc_pr':    _nanmean(auc_pr_vals),
    'mean_cbi':       _nanmean(cbi_vals),
    'mean_brier':     _nanmean(brier_vals),
    'mean_ece':       _nanmean(ece_vals),
}

cols = ['mean_auc_roc','auc_roc_q25','auc_roc_q50','auc_roc_q75',
        'mean_auc_pr','mean_cbi','mean_brier','mean_ece','n_species']

print('\n-- MaskSDM Benchmark (p=1.0, fully unconditioned) --')
for k in cols:
    v = summary[k]
    print(f'  {k:<20} {round(v, 4) if isinstance(v, float) else v}')


-- MaskSDM Benchmark (p=1.0, fully unconditioned) --
  mean_auc_roc         0.8202
  auc_roc_q25          0.7343
  auc_roc_q50          0.8394
  auc_roc_q75          0.9331
  mean_auc_pr          0.1562
  mean_cbi             0.5395
  mean_brier           0.0998
  mean_ece             0.1485
  n_species            151


In [16]:
import json as _json

Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

out_csv  = f'{RESULTS_DIR}/masksdm_benchmark_summary.csv'
out_json = f'{RESULTS_DIR}/masksdm_benchmark_summary.json'

pd.DataFrame([summary]).set_index('masking_p').to_csv(out_csv)
with open(out_json, 'w') as f:
    _json.dump(summary, f, indent=2)

print(f'CSV  saved to {out_csv}')
print(f'JSON saved to {out_json}')

CSV  saved to /content/drive/MyDrive/CISO/eButterfly/results_masksdm_butterfly_1000_epochs_seed1339/masksdm_benchmark_summary.csv
JSON saved to /content/drive/MyDrive/CISO/eButterfly/results_masksdm_butterfly_1000_epochs_seed1339/masksdm_benchmark_summary.json
